In [1]:
# ==========================================
# Notebook 12
# LLM Re-Ranker and Explainer
# ==========================================

import json
import pandas as pd
import numpy as np

from transformers import pipeline

In [2]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [3]:
interactions_df = pd.read_csv("../data/user_interactions.csv")

interactions_df.head()

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4
3,102,1,5
4,102,8,5


In [4]:
hybrid_df = pd.read_csv("../data/hybrid_recommendations.csv")

hybrid_df.head()

,item_id,title,hybrid_score,user_id
0,8,Startup Fundraising Guide,1.556002,101
1,6,Modern Data Engineering,0.590070,101
2,7,Machine Learning in Finance,0.519143,101
3,9,Nutrition Science,0.181014,101
4,4,The Psychology of Habits,0.124730,101


In [5]:
llm = pipeline(
    "text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=300
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vinna\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [7]:
user_id = 101

In [8]:
user_history = interactions_df[interactions_df["user_id"] == user_id]

user_history

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4


In [9]:
history_titles = []

for item_id in user_history["item_id"]:

    title = profiles_df[profiles_df["item_id"] == item_id].iloc[0]["title"]

    history_titles.append(title)

history_titles

['The Early Days of Stripe', 'Building SpaceX', 'Cloud Computing Fundamentals']

In [10]:
candidate_items = hybrid_df[hybrid_df["user_id"] == user_id]

candidate_items

,item_id,title,hybrid_score,user_id
0,8,Startup Fundraising Guide,1.556002,101
1,6,Modern Data Engineering,0.590070,101
2,7,Machine Learning in Finance,0.519143,101
3,9,Nutrition Science,0.181014,101
4,4,The Psychology of Habits,0.124730,101


In [11]:
candidate_text = ""

for _, row in candidate_items.iterrows():

    candidate_text += f"- {row['title']}\n"

candidate_text

'- Startup Fundraising Guide\n- Modern Data Engineering\n- Machine Learning in Finance\n- Nutrition Science\n- The Psychology of Habits\n'

In [12]:
prompt = f"""
You are an expert recommendation engine.

User History:

{history_titles}

Candidate Recommendations:

{candidate_text}

Rank the best 3 recommendations.

Return JSON:

{{
    "recommendations":[
        {{
            "title":"",
            "reason":""
        }}
    ]
}}
"""

In [13]:
print(prompt)


You are an expert recommendation engine.

User History:

['The Early Days of Stripe', 'Building SpaceX', 'Cloud Computing Fundamentals']

Candidate Recommendations:

- Startup Fundraising Guide
- Modern Data Engineering
- Machine Learning in Finance
- Nutrition Science
- The Psychology of Habits


Rank the best 3 recommendations.

Return JSON:

{
    "recommendations":[
        {
            "title":"",
            "reason":""
        }
    ]
}



In [14]:
response = llm(prompt)

In [15]:
response

[{'generated_text': '\nYou are an expert recommendation engine.\n\nUser History:\n\n[\'The Early Days of Stripe\', \'Building SpaceX\', \'Cloud Computing Fundamentals\']\n\nCandidate Recommendations:\n\n- Startup Fundraising Guide\n- Modern Data Engineering\n- Machine Learning in Finance\n- Nutrition Science\n- The Psychology of Habits\n\n\nRank the best 3 recommendations.\n\nReturn JSON:\n\n{\n    "recommendations":[\n        {\n            "title":"",\n            "reason":""\n        }\n    ]\n}\nTo rank these recommendations, I\'ll consider their relevance to the user\'s interests and the depth of content they offer. Let\'s analyze each one:\n\n1. **"The Early Days of Stripe"**:\n   - This is a great historical overview of Stripe\'s early days.\n   - It provides insights into the company\'s founding and early development.\n   - It may be suitable for someone interested in financial technology history or startup culture.\n\n2. **"Building SpaceX"**:\n   - While this is a scientific 

In [16]:
generated_text = response[0]["generated_text"]

generated_text

'\nYou are an expert recommendation engine.\n\nUser History:\n\n[\'The Early Days of Stripe\', \'Building SpaceX\', \'Cloud Computing Fundamentals\']\n\nCandidate Recommendations:\n\n- Startup Fundraising Guide\n- Modern Data Engineering\n- Machine Learning in Finance\n- Nutrition Science\n- The Psychology of Habits\n\n\nRank the best 3 recommendations.\n\nReturn JSON:\n\n{\n    "recommendations":[\n        {\n            "title":"",\n            "reason":""\n        }\n    ]\n}\nTo rank these recommendations, I\'ll consider their relevance to the user\'s interests and the depth of content they offer. Let\'s analyze each one:\n\n1. **"The Early Days of Stripe"**:\n   - This is a great historical overview of Stripe\'s early days.\n   - It provides insights into the company\'s founding and early development.\n   - It may be suitable for someone interested in financial technology history or startup culture.\n\n2. **"Building SpaceX"**:\n   - While this is a scientific exploration of space

In [17]:
print(generated_text)


You are an expert recommendation engine.

User History:

['The Early Days of Stripe', 'Building SpaceX', 'Cloud Computing Fundamentals']

Candidate Recommendations:

- Startup Fundraising Guide
- Modern Data Engineering
- Machine Learning in Finance
- Nutrition Science
- The Psychology of Habits


Rank the best 3 recommendations.

Return JSON:

{
    "recommendations":[
        {
            "title":"",
            "reason":""
        }
    ]
}
To rank these recommendations, I'll consider their relevance to the user's interests and the depth of content they offer. Let's analyze each one:

1. **"The Early Days of Stripe"**:
   - This is a great historical overview of Stripe's early days.
   - It provides insights into the company's founding and early development.
   - It may be suitable for someone interested in financial technology history or startup culture.

2. **"Building SpaceX"**:
   - While this is a scientific exploration of space exploration, it focuses more on science and eng

In [18]:
def generate_recommendation_explanation(user_id):

    user_history = interactions_df[interactions_df["user_id"] == user_id]

    history_titles = []

    for item_id in user_history["item_id"]:

        title = profiles_df[profiles_df["item_id"] == item_id].iloc[0]["title"]

        history_titles.append(title)

    candidates = hybrid_df[hybrid_df["user_id"] == user_id]

    candidate_text = ""

    for _, row in candidates.iterrows():

        candidate_text += f"- {row['title']}\n"

    prompt = f"""
    User History:
    {history_titles}

    Candidate Recommendations:
    {candidate_text}

    Select top 3 recommendations.

    Return JSON:

    {{
        "recommendations":[
            {{
                "title":"",
                "reason":""
            }}
        ]
    }}
    """

    result = llm(prompt)

    return result[0]["generated_text"]

In [19]:
generate_recommendation_explanation(101)

'\n    User History:\n    [\'The Early Days of Stripe\', \'Building SpaceX\', \'Cloud Computing Fundamentals\']\n\n    Candidate Recommendations:\n    - Startup Fundraising Guide\n- Modern Data Engineering\n- Machine Learning in Finance\n- Nutrition Science\n- The Psychology of Habits\n\n\n    Select top 3 recommendations.\n\n    Return JSON:\n\n    {\n        "recommendations":[\n            {\n                "title":"",\n                "reason":""\n            }\n        ]\n    }\n     */\n    public List<Recommendation> getTopThreeRecomendations() {\n        List<String> recommendations = new ArrayList<>();\n        \n        // Add your logic to generate the list of top three recommendations here\n        \n        return new ArrayList<>(recommendations);\n    } \n\n}\n\n```javascript\n\n// Assuming you have a function `getRecommendations` that returns an array of recommendations\nfunction getRecommendations() {\n    // This is just an example. Replace this with actual logic to f

In [20]:
generate_recommendation_explanation(102)

'\n    User History:\n    [\'The Early Days of Stripe\', \'Startup Fundraising Guide\', \'Building SpaceX\']\n\n    Candidate Recommendations:\n    - Cloud Computing Fundamentals\n- Nutrition Science\n- Modern Data Engineering\n- The Psychology of Habits\n- Machine Learning in Finance\n\n\n    Select top 3 recommendations.\n\n    Return JSON:\n\n    {\n        "recommendations":[\n            {\n                "title":"",\n                "reason":""\n            }\n        ]\n    }\n     """\n```\n\nThis code will return a list of the top three recommended books, based on their relevance and popularity. However, please note that this is a fictional example and you should replace it with actual book titles if you have them. Also, make sure to include the correct API endpoint URLs for each book recommendation. Additionally, the code assumes that the data is stored in a JSON file named `book_recommendation.json` which contains an array of book titles and reasons for recommending them.\n

In [21]:
all_explanations = []

In [22]:
for user_id in sorted(hybrid_df["user_id"].unique()):

    result = generate_recommendation_explanation(user_id)

    all_explanations.append({"user_id": user_id, "llm_output": result})

In [23]:
llm_results_df = pd.DataFrame(all_explanations)

llm_results_df.head()

,user_id,llm_output
0,101,\n User History:\n ['The Early Days of S...
1,102,\n User History:\n ['The Early Days of S...
2,103,\n User History:\n ['The Psychology of H...
3,104,"\n User History:\n ['AI for Healthcare',..."
4,105,\n User History:\n ['Cloud Computing Fun...


In [24]:
llm_results_df.to_csv("../data/llm_recommendations.csv", index=False)

In [25]:
saved_df = pd.read_csv("../data/llm_recommendations.csv")

saved_df.head()

,user_id,llm_output
0,101,\n User History:\n ['The Early Days of S...
1,102,\n User History:\n ['The Early Days of S...
2,103,\n User History:\n ['The Psychology of H...
3,104,"\n User History:\n ['AI for Healthcare',..."
4,105,\n User History:\n ['Cloud Computing Fun...


In [26]:
print("Users Processed:", len(llm_results_df))

Users Processed: 5
